# Extract OPIG Antibody Metadata

Each OPIG csv.gz file encodes its metadata as a JSON object in the header row.
This notebook reads that header from every file in the download directory and
collects it into a single metadata CSV.

In [ ]:
import os
import json
import pandas as pd

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
from bcrm.paths import data_path

input_dir   = data_path("opig/raw/")       # written by dataset/01_download_opig.sh
output_file = data_path("metadata/allOPIGdefinedUnpairedMetadata.csv")   # read by build_dataset.Rmd and compare_embeddings.Rmd

In [ ]:
def extract_metadata(filepath):
    """Read the JSON metadata encoded in the header row of an OPIG csv.gz file."""
    header_df = pd.read_csv(filepath, nrows=0)
    header_line = ','.join(header_df.columns)
    metadata = json.loads(header_line)
    metadata["file_name"] = os.path.basename(filepath)
    return metadata


all_files = [f for f in os.listdir(input_dir) if f.endswith(".csv.gz")]
metadata_list = []

for count, filename in enumerate(all_files, start=1):
    metadata_list.append(extract_metadata(os.path.join(input_dir, filename)))
    if count % 100 == 0:
        print(f"{count} / {len(all_files)} files processed")

metadata_df = pd.DataFrame(metadata_list)

# Put file_name first
cols = ["file_name"] + [c for c in metadata_df.columns if c != "file_name"]
metadata_df = metadata_df[cols]

metadata_df.to_csv(output_file, index=False)
print(f"Metadata for {len(metadata_df)} files written to {output_file}")